![Du Bois Does Data — Enhanced Visualizations](https://raw.githubusercontent.com/ShdwSpde/dubois-does-data-notebooks/main/assets/header-06.png)

![The headline](https://raw.githubusercontent.com/ShdwSpde/dubois-does-data-notebooks/main/assets/key-06.png)


## Data Sources & Provenance

> **Open Data Week Notice:** This notebook visualizes data from Notebooks 1-5. All underlying data is derived from publicly available U.S. government sources. See individual notebooks for detailed source references.

| Data | Source | Reference |
|------|--------|-----------|
| All metrics (population, education, occupation, income, wealth) | Synthesized from Notebooks 1-4 | See provenance cells in each notebook |
| Du Bois visual style reference | W.E.B. Du Bois, 1900 Paris Exposition | [Library of Congress](https://www.loc.gov/pictures/collection/anedub/) |

**Methodology notes:**
- **Education column changes definition mid-series:** 1900-1925 = literacy rate; 1950+ = HS completion. The visual trend line is not a continuous measure
- **1925 and 1975 are interpolated** from adjacent Census years, not direct measurements
- **"2025" label represents 2022-2023 data** (most recent available)
- Wealth parity pre-1989 and income parity pre-1967 are historical estimates
- The $4.1T wealth gap figure: ($285,000 - $44,900) × 17.2M Black households (2022 Fed SCF + Census ACS)

# Enhanced Visualizations: Interactive, Aesthetic, and Animated
## Bringing Du Bois's Data to Life with Modern Techniques

This notebook implements three advanced visualization enhancements:
1. **Interactive Dashboard** - Explore data with hover, zoom, and filters
2. **Du Bois Aesthetic Recreations** - Honor his visual style with modern data
3. **Animated Time-Lapses** - Watch 125 years of change unfold

These visualizations make the data more engaging, shareable, and impactful for presentations, social media, and grant proposals.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle, Wedge, FancyBboxPatch
from matplotlib.gridspec import GridSpec
import matplotlib.animation as animation
from IPython.display import HTML

# Try to import plotly for interactive visualizations
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
    print("✓ Plotly available - Interactive dashboards enabled")
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠ Plotly not installed. Run: pip install plotly")
    print("  Interactive dashboards will be skipped.")

# Du Bois color palette
dubois_colors = {
    'crimson': '#DC143C',
    'gold': '#FFD700', 
    'dark_slate': '#2F4F4F',
    'brown': '#8B4513',
    'black': '#000000',
    'dark_green': '#2b5329',
    'burgundy': '#654321'
}

print("\n" + "="*80)
print("ENHANCED VISUALIZATIONS TOOLKIT LOADED")
print("="*80)

## Load Data from Previous Notebooks

In [ ]:
# Recreate the consolidated progress data (from Notebook 5)
# Sources: Census Bureau, BLS, Fed SCF — see individual notebooks for details
# Note: 1925/1975 are interpolated; "2025" = 2022-23 actuals; Education metric changes at 1950
progress_data = pd.DataFrame({
    'Year': [1900, 1925, 1950, 1975, 2000, 2025],
    'Year_Label': ['1900', '1925*', '1950', '1975*', '2000', '2022-23'],
    'Population_Pct': [11.6, 9.3, 9.9, 11.5, 12.9, 12.6],
    'Education_Pct': [55, 60, 20.1, 51.2, 78.5, 90.1],  # CAUTION: Literacy (1900-25) → HS Completion (1950+)
    'Professional_Pct': [1, 2, 5, 14, 25.5, 36],
    'Income_Parity': [None, None, 62, 61, 62, 65],
    'Wealth_Parity': [None, None, 17, 15, 17, 16],
    'Era': ['Jim Crow', 'Great Migration', 'Post-WWII', 'Post-Civil Rights', 'New Millennium', 'Present Day'],
    'Data_Note': ['Census/Du Bois', 'Interpolated', 'Census/BLS/SCF', 'Interpolated', 'Census/BLS/SCF', '2022-23 actuals']
})

print("Consolidated Progress Data (125 Years):")
print("⚠ * = Interpolated | Education: Literacy pre-1950, HS Completion post-1950 | 2025 = 2022-23 actuals")
print(progress_data[['Year_Label', 'Population_Pct', 'Education_Pct', 'Professional_Pct', 
                      'Income_Parity', 'Wealth_Parity', 'Era']].to_string(index=False))

---

# ENHANCEMENT #2: Interactive Dashboard (Plotly)

Create web-based interactive visualizations that allow users to:
- Hover over data points for detailed information
- Zoom and pan to explore specific time periods
- Toggle metrics on/off to compare specific indicators
- Export as standalone HTML files for presentations

In [ ]:
if PLOTLY_AVAILABLE:
    print("\n" + "="*80)
    print("CREATING INTERACTIVE DASHBOARD")
    print("="*80)
    
    # Create interactive multi-metric dashboard
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Education Progress (Literacy → HS Completion)',
            'Professional Occupations Growth',
            'Income Parity (% of White Income)',
            'Wealth Parity (% of White Wealth)'
        ),
        specs=[[{}, {}], [{}, {}]],
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )
    
    # Panel 1: Education
    fig.add_trace(
        go.Scatter(
            x=progress_data['Year'],
            y=progress_data['Education_Pct'],
            mode='lines+markers',
            name='Education',
            line=dict(color=dubois_colors['crimson'], width=3),
            marker=dict(size=12, line=dict(width=2, color='black')),
            hovertemplate='<b>%{x}</b><br>' +
                         'Education: %{y:.1f}%<br>' +
                         'Era: %{customdata}<br>' +
                         '<extra></extra>',
            customdata=progress_data['Era']
        ),
        row=1, col=1
    )
    
    # Panel 2: Professional
    fig.add_trace(
        go.Scatter(
            x=progress_data['Year'],
            y=progress_data['Professional_Pct'],
            mode='lines+markers',
            name='Professional',
            line=dict(color=dubois_colors['gold'], width=3),
            marker=dict(size=12, symbol='square', line=dict(width=2, color='black')),
            hovertemplate='<b>%{x}</b><br>' +
                         'Professional: %{y:.1f}%<br>' +
                         'Era: %{customdata}<br>' +
                         '<extra></extra>',
            customdata=progress_data['Era']
        ),
        row=1, col=2
    )
    
    # Panel 3: Income Parity
    income_data = progress_data[progress_data['Income_Parity'].notna()]
    fig.add_trace(
        go.Scatter(
            x=income_data['Year'],
            y=income_data['Income_Parity'],
            mode='lines+markers',
            name='Income Parity',
            line=dict(color=dubois_colors['dark_slate'], width=3),
            marker=dict(size=12, symbol='diamond', line=dict(width=2, color='black')),
            hovertemplate='<b>%{x}</b><br>' +
                         'Income Parity: %{y:.0f}%<br>' +
                         'Gap: %{customdata:.0f} points from parity<br>' +
                         '<extra></extra>',
            customdata=100 - income_data['Income_Parity']
        ),
        row=2, col=1
    )
    # Add parity line
    fig.add_hline(y=100, line_dash="dash", line_color="black", 
                 annotation_text="Parity", row=2, col=1)
    
    # Panel 4: Wealth Parity
    wealth_data = progress_data[progress_data['Wealth_Parity'].notna()]
    fig.add_trace(
        go.Scatter(
            x=wealth_data['Year'],
            y=wealth_data['Wealth_Parity'],
            mode='lines+markers',
            name='Wealth Parity',
            line=dict(color=dubois_colors['brown'], width=3),
            marker=dict(size=15, symbol='star', line=dict(width=2, color='black')),
            hovertemplate='<b>%{x}</b><br>' +
                         'Wealth Parity: %{y:.0f}%<br>' +
                         'Gap: %{customdata:.0f} points from parity<br>' +
                         '<b>STUCK FOR 75 YEARS</b><br>' +
                         '<extra></extra>',
            customdata=100 - wealth_data['Wealth_Parity']
        ),
        row=2, col=2
    )
    # Add parity line
    fig.add_hline(y=100, line_dash="dash", line_color="black",
                 annotation_text="Parity", row=2, col=2)
    
    # Update layout
    fig.update_xaxes(title_text="Year", gridcolor='lightgray')
    fig.update_yaxes(title_text="Percentage", gridcolor='lightgray')
    
    fig.update_layout(
        title=dict(
            text="<b>Interactive Progress Dashboard: 1900-2025</b><br>" +
                 "<sub>Hover for details | Click legend to toggle | Drag to zoom</sub>",
            x=0.5,
            xanchor='center',
            font=dict(size=20)
        ),
        height=800,
        showlegend=True,
        hovermode='closest',
        plot_bgcolor='white',
        font=dict(family="Arial, sans-serif", size=12)
    )
    
    # Save as standalone HTML
    fig.write_html('interactive_dashboard.html')
    print("\n✓ Interactive dashboard saved: interactive_dashboard.html")
    print("  Open in browser for full interactivity")
    
    # Display in notebook
    fig.show()
    
else:
    print("\n⚠ Skipping interactive dashboard - Plotly not available")
    print("  Install with: pip install plotly")

In [ ]:
if PLOTLY_AVAILABLE:
    print("\n" + "="*80)
    print("CREATING INTERACTIVE WEALTH GAP VISUALIZATION")
    print("="*80)
    
    # Create dramatic wealth gap visualization
    fig = go.Figure()
    
    # Add wealth parity over time
    wealth_data = progress_data[progress_data['Wealth_Parity'].notna()].copy()
    
    fig.add_trace(go.Scatter(
        x=wealth_data['Year'],
        y=wealth_data['Wealth_Parity'],
        mode='lines+markers',
        name='Black Wealth (% of White)',
        line=dict(color=dubois_colors['crimson'], width=4),
        marker=dict(size=20, line=dict(width=3, color='black')),
        fill='tozeroy',
        fillcolor='rgba(220, 20, 60, 0.3)',
        hovertemplate='<b>Year: %{x}</b><br>' +
                     'Black Wealth: %{y:.0f}% of white<br>' +
                     'Gap: %{customdata:.0f} points<br>' +
                     '<b>$4.1 TRILLION MISSING</b><br>' +
                     '<extra></extra>',
        customdata=100 - wealth_data['Wealth_Parity']
    ))
    
    # Add parity line
    fig.add_hline(
        y=100, 
        line_dash="dash", 
        line_color="green", 
        line_width=3,
        annotation=dict(
            text="<b>PARITY (100%)</b>",
            font=dict(size=14, color="green"),
            showarrow=False
        )
    )
    
    # Add annotations for key years
    fig.add_annotation(
        x=1950, y=17,
        text="<b>1950: 17%</b><br>Starting point",
        showarrow=True,
        arrowhead=2,
        ax=0, ay=-40,
        font=dict(size=12)
    )
    
    fig.add_annotation(
        x=2025, y=16,
        text="<b>2025: 16%</b><br>ZERO PROGRESS<br>in 75 years",
        showarrow=True,
        arrowhead=2,
        ax=0, ay=40,
        font=dict(size=12, color='darkred')
    )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text="<b>The Wealth Gap: 75 Years of Stagnation</b><br>" +
                 "<sub>Black wealth stuck at 16% of white wealth = $4.1 TRILLION missing</sub>",
            x=0.5,
            xanchor='center',
            font=dict(size=22)
        ),
        xaxis=dict(
            title="<b>Year</b>",
            gridcolor='lightgray',
            range=[1945, 2030]
        ),
        yaxis=dict(
            title="<b>Black Wealth as % of White Wealth</b>",
            gridcolor='lightgray',
            range=[0, 110]
        ),
        height=600,
        plot_bgcolor='white',
        font=dict(family="Arial, sans-serif", size=14),
        hovermode='x unified'
    )
    
    # Save
    fig.write_html('interactive_wealth_gap.html')
    print("\n✓ Interactive wealth gap saved: interactive_wealth_gap.html")
    
    # Display
    fig.show()

---

# ENHANCEMENT #5: Du Bois Aesthetic Recreations

Recreate W.E.B. Du Bois's iconic 1900 visual style using modern data:
- Hand-drawn aesthetic
- Spiral and radial charts
- His exact color palette
- French text labels (Paris Exposition)
- Ornate borders and typography

In [ ]:
print("\n" + "="*80)
print("DU BOIS AESTHETIC RECREATION #1: SPIRAL CHART")
print("Recreating his famous spiral wealth accumulation chart")
print("="*80)

# Use hand-drawn style
plt.xkcd()

# Create figure with vintage parchment background
fig, ax = plt.subplots(figsize=(12, 14), facecolor='#f5f5dc')
ax.set_facecolor('#f5f5dc')

# Data for spiral: Wealth growth over time
years_spiral = [1900, 1925, 1950, 1975, 2000, 2025]
# Using professional % as proxy for economic advancement
values_spiral = [1, 2, 5, 14, 25.5, 36]

# Create spiral coordinates
theta = np.linspace(0, 4 * np.pi, 100)
r = np.linspace(0.5, 4, 100)

# Plot spiral
ax.plot(r * np.cos(theta), r * np.sin(theta), 
        color=dubois_colors['burgundy'], linewidth=4, alpha=0.3)

# Add data points on spiral
num_points = len(years_spiral)
point_positions = np.linspace(0, len(theta)-1, num_points, dtype=int)

for i, (year, value, pos) in enumerate(zip(years_spiral, values_spiral, point_positions)):
    x = r[pos] * np.cos(theta[pos])
    y = r[pos] * np.sin(theta[pos])
    
    # Plot point
    ax.scatter(x, y, s=value*30, color=dubois_colors['crimson'], 
              edgecolor='black', linewidth=2, zorder=3)
    
    # Add label
    offset_x = 0.5 if x > 0 else -0.5
    offset_y = 0.3 if y > 0 else -0.3
    
    ax.annotate(f"{year}\n{value}%", 
               xy=(x, y), 
               xytext=(x + offset_x, y + offset_y),
               fontsize=11, fontweight='bold',
               ha='center', va='center',
               bbox=dict(boxstyle='round,pad=0.3', 
                        facecolor=dubois_colors['gold'], 
                        edgecolor='black', linewidth=2))

# Add ornate title (French style like Du Bois)
title_text = "PROGRÈS PROFESSIONNEL\nDES NÈGRES AMÉRICAINS\n1900-2025"
ax.text(0, 5.2, title_text, 
        ha='center', va='top', fontsize=18, fontweight='bold',
        family='serif',
        bbox=dict(boxstyle='round,pad=0.8', 
                 facecolor=dubois_colors['dark_green'], 
                 edgecolor='black', linewidth=3,
                 alpha=0.9),
        color='white')

# Add subtitle in English
subtitle = "Professional Advancement of Black Americans\nFrom 1% (1900) to 36% (2025)"
ax.text(0, -5.2, subtitle,
        ha='center', va='bottom', fontsize=11, style='italic',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                 edgecolor='black', linewidth=2))

# Add Du Bois attribution
ax.text(-4.5, -4.5, "Inspiré par W.E.B. Du Bois\nExposition Universelle, Paris 1900",
        fontsize=8, style='italic', ha='left', va='bottom')

# Remove axes
ax.set_xlim(-5, 5)
ax.set_ylim(-5.5, 5.5)
ax.axis('off')

# Add decorative border
border = Rectangle((-4.8, -5.3), 9.6, 10.6, 
                   fill=False, edgecolor='black', linewidth=4)
ax.add_patch(border)

plt.tight_layout()
plt.savefig('dubois_spiral_recreation.png', dpi=300, facecolor='#f5f5dc', bbox_inches='tight')
plt.show()

print("\n✓ Du Bois spiral chart saved: dubois_spiral_recreation.png")

# Reset matplotlib style
plt.rcdefaults()

In [ ]:
print("\n" + "="*80)
print("DU BOIS AESTHETIC RECREATION #2: RADIAL BAR CHART")
print("Showing occupational transformation in Du Bois style")
print("="*80)

# Use hand-drawn style
plt.xkcd()

fig, ax = plt.subplots(figsize=(14, 14), subplot_kw=dict(projection='polar'),
                       facecolor='#f5f5dc')
ax.set_facecolor('#f5f5dc')

# Data
categories = ['1900', '1925', '1950', '1975', '2000', '2025']
professional = [1, 2, 5, 14, 25.5, 36]

# Number of variables
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
professional += professional[:1]  # Complete the circle
angles += angles[:1]

# Plot
ax.plot(angles, professional, 'o-', linewidth=4, 
        color=dubois_colors['crimson'], markersize=15,
        markeredgecolor='black', markeredgewidth=2)
ax.fill(angles, professional, alpha=0.25, color=dubois_colors['gold'])

# Fix axis to go in the right order
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Draw axis lines for each angle and label
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=14, fontweight='bold')

# Set y-axis limits
ax.set_ylim(0, 40)
ax.set_yticks([10, 20, 30, 40])
ax.set_yticklabels(['10%', '20%', '30%', '40%'], fontsize=11)

# Add title in French
title_text = "OCCUPATIONS PROFESSIONNELLES\nCROISSANCE SUR 125 ANS"
plt.title(title_text, 
         fontsize=20, fontweight='bold', pad=30, family='serif',
         bbox=dict(boxstyle='round,pad=0.8', 
                  facecolor=dubois_colors['dark_green'],
                  edgecolor='black', linewidth=3),
         color='white')

# Add subtitle
fig.text(0.5, 0.08, 
         "Professional & Management Occupations: 1% → 36% (36x increase)\n" +
         "Dans le style de W.E.B. Du Bois",
         ha='center', fontsize=12, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                  edgecolor='black', linewidth=2))

# Grid styling
ax.grid(True, color='black', linewidth=1.5, alpha=0.3)

plt.tight_layout()
plt.savefig('dubois_radial_recreation.png', dpi=300, facecolor='#f5f5dc', bbox_inches='tight')
plt.show()

print("\n✓ Du Bois radial chart saved: dubois_radial_recreation.png")

# Reset
plt.rcdefaults()

In [ ]:
print("\n" + "="*80)
print("DU BOIS AESTHETIC RECREATION #3: COMPARATIVE BAR CHART")
print("Then vs Now in authentic Du Bois style")
print("="*80)

# Hand-drawn style
plt.xkcd()

fig, ax = plt.subplots(figsize=(12, 10), facecolor='#f5f5dc')
ax.set_facecolor('#f5f5dc')

# Data
metrics = ['ÉDUCATION\n(Alphabétisation)', 'PROFESSIONNEL\n(% de la force)', 
           'RICHESSE\n(Parité %)']
values_1900 = [55, 1, 0]  # Wealth was essentially 0 in parity terms
values_2025 = [90, 36, 16]

x = np.arange(len(metrics))
width = 0.35

# Create bars
bars1 = ax.barh(x - width/2, values_1900, width, 
                label='1900', color=dubois_colors['burgundy'],
                edgecolor='black')
bars2 = ax.barh(x + width/2, values_2025, width,
                label='2025', color=dubois_colors['gold'],
                edgecolor='black')

# Labels
ax.set_yticks(x)
ax.set_yticklabels(metrics, fontsize=14, fontweight='bold', family='serif')
ax.set_xlabel('POURCENTAGE', fontsize=14, fontweight='bold', family='serif')

# Add values on bars
for bars, values in [(bars1, values_1900), (bars2, values_2025)]:
    for bar, value in zip(bars, values):
        if value > 0:
            ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                   f'{value:.0f}%', va='center', fontsize=12, fontweight='bold')

# Title in French
title = "COMPARAISON DES PROGRÈS\nNÈGRES AMÉRICAINS\n1900-2025"
ax.text(50, 3.2, title,
        ha='center', va='center', fontsize=20, fontweight='bold',
        family='serif',
        bbox=dict(boxstyle='round,pad=1', 
                 facecolor=dubois_colors['dark_green'],
                 edgecolor='black', linewidth=4),
        color='white')

# Legend
ax.legend(loc='lower right', fontsize=13, frameon=True,
         fancybox=True, shadow=True, edgecolor='black')

# Grid
ax.grid(True, axis='x', color='black', linewidth=1, alpha=0.3)
ax.set_xlim(0, 100)

# Decorative border
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(4)

# Attribution
ax.text(2, -0.7, "D'après les données de W.E.B. Du Bois\nActualisé pour 2025",
        fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('dubois_comparison_recreation.png', dpi=300, facecolor='#f5f5dc', bbox_inches='tight')
plt.show()

print("\n✓ Du Bois comparison chart saved: dubois_comparison_recreation.png")

# Reset
plt.rcdefaults()

---

# ENHANCEMENT #8: Animated Time-Lapses

Create animated visualizations showing change over time:
- Watch professional jobs grow from 1% to 36%
- See the wealth gap remain frozen for 75 years
- Observe education climb from literacy to universal HS

**Note:** Animations may take 1-2 minutes to generate

In [ ]:
print("\n" + "="*80)
print("CREATING ANIMATION #1: PROFESSIONAL JOBS GROWTH (BAR CHART RACE)")
print("This may take 1-2 minutes...")
print("="*80)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')

# Create figure
fig, ax = plt.subplots(figsize=(12, 8))

# Data
years = [1900, 1925, 1950, 1975, 2000, 2025]
professional_pct = [1, 2, 5, 14, 25.5, 36]

# Create smooth interpolation for animation
from scipy.interpolate import interp1d
f = interp1d(years, professional_pct, kind='cubic')
years_smooth = np.linspace(1900, 2025, 60)  # 60 frames
values_smooth = f(years_smooth)

# Animation function
def animate(frame):
    ax.clear()
    
    current_year = years_smooth[frame]
    current_value = values_smooth[frame]
    
    # Create bar
    bar = ax.barh(['Professional\nOccupations'], [current_value],
                   color=dubois_colors['gold'], 
                   edgecolor='black', linewidth=3)
    
    # Add value label
    ax.text(current_value + 1, 0, f'{current_value:.1f}%',
           va='center', fontsize=24, fontweight='bold')
    
    # Styling
    ax.set_xlim(0, 40)
    ax.set_xlabel('Percentage of Black Workforce', fontsize=16, fontweight='bold')
    
    # Title with year
    ax.set_title(f'Professional & Management Occupations\nBlack Americans: {int(current_year)}',
                fontsize=20, fontweight='bold', pad=20)
    
    # Add growth annotation
    growth = ((current_value / 1) - 1) * 100 if current_value > 0 else 0
    ax.text(20, -0.35, f'{growth:.0f}% growth since 1900',
           ha='center', fontsize=14, style='italic',
           bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    
    # Grid
    ax.grid(True, axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    
    return ax,

# Create animation
anim = animation.FuncAnimation(fig, animate, frames=len(years_smooth),
                              interval=100, blit=False, repeat=True)

# Save as GIF
try:
    anim.save('professional_growth_animation.gif', writer='pillow', fps=10)
    print("\n✓ Animation saved: professional_growth_animation.gif")
except Exception as e:
    print(f"\n⚠ Could not save GIF: {e}")
    print("  Install pillow: pip install pillow")

# Display in notebook
plt.close()
HTML(anim.to_jshtml())

In [ ]:
print("\n" + "="*80)
print("CREATING ANIMATION #2: THE WEALTH GAP (STAGNATION)")
print("Showing 75 years of near-zero movement")
print("="*80)

fig, ax = plt.subplots(figsize=(14, 8))

# Data
wealth_years = [1950, 1975, 2000, 2025]
wealth_parity = [17, 15, 17, 16]

# Smooth interpolation
f_wealth = interp1d(wealth_years, wealth_parity, kind='linear')
wealth_smooth_years = np.linspace(1950, 2025, 75)  # 75 frames for 75 years
wealth_smooth_values = f_wealth(wealth_smooth_years)

def animate_wealth(frame):
    ax.clear()
    
    # Show data up to current frame
    current_years = wealth_smooth_years[:frame+1]
    current_values = wealth_smooth_values[:frame+1]
    current_year = wealth_smooth_years[frame]
    current_value = wealth_smooth_values[frame]
    
    # Plot line
    ax.plot(current_years, current_values, 
           color=dubois_colors['crimson'], linewidth=4,
           marker='o', markersize=10, markeredgecolor='black', markeredgewidth=2)
    
    # Fill area
    ax.fill_between(current_years, 0, current_values,
                    alpha=0.3, color=dubois_colors['crimson'])
    
    # Parity line
    ax.axhline(y=100, color='green', linestyle='--', linewidth=3, alpha=0.7,
              label='Parity (100%)')
    
    # Current value annotation
    ax.scatter([current_year], [current_value], s=500, 
              color='yellow', edgecolor='black', linewidth=3, zorder=5)
    ax.text(current_year, current_value + 8, 
           f'{int(current_year)}: {current_value:.0f}%',
           ha='center', fontsize=16, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='yellow', edgecolor='black', linewidth=2))
    
    # Styling
    ax.set_xlim(1945, 2030)
    ax.set_ylim(0, 110)
    ax.set_xlabel('Year', fontsize=16, fontweight='bold')
    ax.set_ylabel('Black Wealth as % of White Wealth', fontsize=16, fontweight='bold')
    ax.set_title('The Wealth Gap: 75 Years of Stagnation\n16-17% for Three Generations',
                fontsize=20, fontweight='bold', pad=20, color='darkred')
    
    # Add gap annotation
    gap = 100 - current_value
    ax.text(1950, 50, f'GAP: {gap:.0f} points\n$4.1 TRILLION MISSING',
           fontsize=18, fontweight='bold', color='darkred',
           bbox=dict(boxstyle='round', facecolor='lightcoral', 
                    edgecolor='darkred', linewidth=3, alpha=0.9))
    
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=14)
    
    return ax,

# Create animation
anim_wealth = animation.FuncAnimation(fig, animate_wealth, 
                                     frames=len(wealth_smooth_years),
                                     interval=50, blit=False, repeat=True)

# Save
try:
    anim_wealth.save('wealth_gap_stagnation.gif', writer='pillow', fps=15)
    print("\n✓ Animation saved: wealth_gap_stagnation.gif")
except Exception as e:
    print(f"\n⚠ Could not save GIF: {e}")

plt.close()
HTML(anim_wealth.to_jshtml())

In [ ]:
print("\n" + "="*80)
print("CREATING ANIMATION #3: MULTI-METRIC RACE (ALL TOGETHER)")
print("Watch all four metrics progress simultaneously")
print("="*80)

fig, ax = plt.subplots(figsize=(14, 10))

# Data for all metrics (normalized to 0-100 for comparison)
years_all = [1900, 1925, 1950, 1975, 2000, 2025]
education_norm = [0, 5, 20, 51, 78, 90]
professional_norm = [0, 2, 5, 14, 25.5, 36]

# Interpolate
years_smooth_all = np.linspace(1900, 2025, 100)
f_edu = interp1d(years_all, education_norm, kind='cubic')
f_pro = interp1d(years_all, professional_norm, kind='cubic')
edu_smooth = f_edu(years_smooth_all)
pro_smooth = f_pro(years_smooth_all)

def animate_multi(frame):
    ax.clear()
    
    current_years = years_smooth_all[:frame+1]
    current_edu = edu_smooth[:frame+1]
    current_pro = pro_smooth[:frame+1]
    
    # Plot both lines
    ax.plot(current_years, current_edu, 
           color=dubois_colors['crimson'], linewidth=4, 
           label='Education (Literacy→HS)', marker='o', markersize=8)
    ax.plot(current_years, current_pro,
           color=dubois_colors['gold'], linewidth=4,
           label='Professional Jobs', marker='s', markersize=8)
    
    # Current year marker
    current_year = years_smooth_all[frame]
    ax.axvline(x=current_year, color='black', linestyle='--', linewidth=2, alpha=0.5)
    
    # Year display
    ax.text(0.98, 0.98, f'{int(current_year)}',
           transform=ax.transAxes, fontsize=60, fontweight='bold',
           ha='right', va='top', alpha=0.3)
    
    # Styling
    ax.set_xlim(1895, 2030)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Year', fontsize=16, fontweight='bold')
    ax.set_ylabel('Percentage', fontsize=16, fontweight='bold')
    ax.set_title('125 Years of Progress: The Race to Advancement',
                fontsize=20, fontweight='bold', pad=20)
    
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=14, frameon=True)
    
    return ax,

# Create animation
anim_multi = animation.FuncAnimation(fig, animate_multi,
                                    frames=len(years_smooth_all),
                                    interval=50, blit=False, repeat=True)

# Save
try:
    anim_multi.save('multi_metric_race.gif', writer='pillow', fps=20)
    print("\n✓ Animation saved: multi_metric_race.gif")
except Exception as e:
    print(f"\n⚠ Could not save GIF: {e}")

plt.close()
HTML(anim_multi.to_jshtml())

---

## Summary: Enhanced Visualizations Created

### ENHANCEMENT #2: Interactive Dashboard
✓ **interactive_dashboard.html** - 4-panel interactive dashboard  
✓ **interactive_wealth_gap.html** - Dramatic wealth gap visualization  
- Hover for details
- Zoom and pan
- Toggle metrics
- Share as standalone HTML

### ENHANCEMENT #5: Du Bois Aesthetic Recreations  
✓ **dubois_spiral_recreation.png** - Spiral growth chart (French labels)  
✓ **dubois_radial_recreation.png** - Radial bar chart  
✓ **dubois_comparison_recreation.png** - Then vs Now comparison  
- Hand-drawn style (xkcd mode)
- Vintage parchment background
- French text (Paris Exposition style)
- Ornate borders and typography

### ENHANCEMENT #8: Animated Time-Lapses  
✓ **professional_growth_animation.gif** - Watch 1% → 36% growth  
✓ **wealth_gap_stagnation.gif** - 75 years frozen at 16%  
✓ **multi_metric_race.gif** - All metrics racing together  
- Social media ready
- Presentation friendly
- Emotionally engaging

---

## Usage Recommendations

**For Grant Proposals:**
- Use interactive HTML files for live presentations
- Include Du Bois recreations to show historical continuity
- Share wealth gap animation to create urgency

**For Social Media:**
- Post GIF animations (auto-play, eye-catching)
- Share Du Bois aesthetic recreations (unique, beautiful)
- Link to interactive dashboards for deep dives

**For Academic Presentations:**
- Show Du Bois recreations to honor methodology
- Use interactive dashboards for Q&A exploration
- Include animations to demonstrate temporal patterns

---

## Technical Notes

**Requirements:**
```bash
pip install plotly pillow scipy
```

**File Sizes:**
- HTML files: ~500KB each (self-contained)
- PNG files: ~2-3MB each (high resolution)
- GIF files: ~1-2MB each (60-100 frames)

**Browser Compatibility:**
- Interactive HTML works in all modern browsers
- No internet connection required (self-contained)
- Mobile-friendly responsive design

---

*"Data reveals truth beyond rhetoric. Visualization makes truth undeniable."*  
— Inspired by W.E.B. Du Bois

![Du Bois Does Data — sources: Library of Congress, U.S. Census, BLS](https://raw.githubusercontent.com/ShdwSpde/dubois-does-data-notebooks/main/assets/footer.png)
